In [1]:
import sys
import os 

path_to_src = "../src"

if path_to_src not in sys.path:
    sys.path.append(os.path.abspath(path_to_src))
    
from model import *
from utils import *
from metrics import *

In [2]:
def simulation_2_custom(pi_levels=[0.1, 0.2], configs=None, n_runs=30, correlated=False):
    """
    Hàm chạy Simulation 2 với tùy chọn tỉ lệ outlier và loại hàm ngưỡng.
    configs: list các tuple [("soft", "soft"), ("scad", "scad"), ...]
    """
    if configs is None:
        configs = [("soft", "soft"), ("scad", "scad")]
    
    all_results = {}
    print(f"=== START SIMULATION 2 (Correlated={correlated}) ===")

    for pi in pi_levels:
        print(f"\nProcessing π = {pi}...")
        config_data = {f"{e}-{w}": [] for e, w in configs}

        for run in range(n_runs):
            X, y, true_outliers, _ = generate_data_sim2(
                contamination=pi, correlated=correlated, random_state=run
            )

            for thE, thW in configs:
                name = f"{thE}-{thW}"
                
                l1, l2 = select_lambda(X, thresh_E=thE, thresh_w=thW)
                

                res = arsk(
                    X, n_clusters=3, lambda1=l1, lambda2=l2,
                    thresh_E=thE, thresh_w=thW, random_state=run
                )
                
                cer = compute_cer(y, res["labels"], res["errors"], true_outliers, 3)
                config_data[name].append(cer)

        summary = {}
        for name, cers in config_data.items():
            summary[name] = {
                "mean": np.mean(cers),
                "se": np.std(cers) / np.sqrt(n_runs)
            }
        all_results[pi] = summary

    return all_results

In [3]:
my_configs = [("soft", "soft"), ("scad", "scad")]
results_sim2 = simulation_2_custom(
    pi_levels=[0, 0.1, 0.2], 
    configs=my_configs, 
    n_runs=100,
    correlated=True
)

for pi, data in results_sim2.items():
    print(f"\n--- Results for π = {pi} ---")
    for name, val in data.items():
        print(f"{name}: CER = {val['mean']:.4f} (±{val['se']:.4f})")

=== START SIMULATION 2 (Correlated=True) ===

Processing π = 0...


KeyboardInterrupt: 